# QQQI / QQQ / TQQQ v4.1 complete backtest review

Strategy: `qqqi_qqq_tqqq_vxn_leverage_v4_1`  
Evidence boundary: 2026-07-31  
Status: research-only, not trade-ready, post-result hypothesis.

This notebook reproduces the full three-asset backtest, plots close-derived signals and next-open executed buys/sells, evaluates every transition over 5/10/20/40 sessions, and then runs the 2010-2026 QQQ/TQQQ attack-layer validation without backfilling QQQI.

## Research interpretation

- QQQ price repair identifies recovery opportunity.
- VIX controls broad-market defense and the initial QQQ risk-on state.
- VXN only vetoes the 75% TQQQ layer.
- State 0 is 100% QQQI, state 1 is 100% QQQ, and state 2 is 25% QQQ plus 75% TQQQ.
- Signals are decided at close and executed at the next adjusted open.
- Transaction cost is 10 bps per turnover unit.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from IPython.display import display, Markdown

from src.research.etf_rotation_experiment import (
    chronological_split_metrics,
    fetch_adjusted_daily_bars,
)
from src.research.vxn_attack_layer_long_history import run_attack_layer_comparison
from src.research.vxn_leverage_overlay_experiment import run_vxn_leverage_overlay_comparison

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 180)
END_DATE = '2026-08-01'
PRIMARY = 'rotation_vxn_leverage_v4_1_75'
BASELINE = 'rotation_vix_v3_75'


In [ ]:
primary_contract_path = Path('../configs/research_paradigms/qqqi_qqq_tqqq_vxn_leverage_v4_1.yaml')
primary_contract = yaml.safe_load(primary_contract_path.read_text(encoding='utf-8'))
primary_symbols = ['QQQI', 'QQQ', 'TQQQ', '^VIX', '^VXN']
primary_bars, primary_coverage = fetch_adjusted_daily_bars(
    symbols=primary_symbols,
    start=primary_contract['data']['start_date'],
    end=END_DATE,
)
primary_metrics, primary_results, prepared, diagnostics = run_vxn_leverage_overlay_comparison(
    primary_bars, primary_contract
)
display(primary_coverage)
display(primary_metrics[[
    'total_return', 'cagr', 'annual_volatility', 'sharpe', 'sortino',
    'max_drawdown', 'calmar', 'switch_count', 'turnover_units',
    'transaction_cost_paid', 'pct_time_qqqi', 'pct_time_qqq',
    'pct_time_partial_tqqq', 'average_tqqq_weight'
]])


## Full portfolio result

Published v4.1 result: total return 101.17%, CAGR 32.44%, volatility 25.82%, Sharpe 1.218, Sortino 1.764, maximum drawdown -24.43%, and Calmar 1.328. The VXN overlay uses fewer leveraged sessions than VIX v3 but produces a higher cumulative leveraged-state return.

In [ ]:
names = {
    'buy_hold_QQQ': 'QQQ buy and hold',
    BASELINE: 'VIX v3, 75% TQQQ',
    PRIMARY: 'v4.1 VXN leverage veto',
}
equity = pd.concat(
    {names[key]: result.daily['equity'] for key, result in primary_results.items() if key in names},
    axis=1,
)
ax = equity.plot(figsize=(13, 5), title='Full three-asset backtest: equity')
ax.set_ylabel('Equity')
ax.grid(True, alpha=0.3)
plt.show()
drawdown = equity.div(equity.cummax()).sub(1.0)
ax = drawdown.plot(figsize=(13, 4), title='Full three-asset backtest: drawdown')
ax.set_ylabel('Drawdown')
ax.grid(True, alpha=0.3)
plt.show()


In [ ]:
overlay = primary_results[PRIMARY].daily.copy()
baseline = primary_results[BASELINE].daily.copy()
state_names = {0: 'QQQI defensive', 1: 'QQQ attack', 2: '25% QQQ + 75% TQQQ'}
action_names = {
    (0, 1): 'Enter QQQ',
    (1, 2): 'Add TQQQ leverage',
    (2, 1): 'Reduce TQQQ leverage',
    (1, 0): 'Move to QQQI defense',
    (2, 0): 'Move to QQQI defense',
}
trade_mask = overlay['position_state'].ne(overlay['position_state'].shift())
execution_rows = overlay.loc[trade_mask].iloc[1:]
records = []
for execution_date, execution_row in execution_rows.iterrows():
    location = overlay.index.get_loc(execution_date)
    prior_row = overlay.iloc[location - 1]
    from_state = int(prior_row['position_state'])
    to_state = int(execution_row['position_state'])
    record = {
        'signal_date': overlay.index[location - 1],
        'execution_date': execution_date,
        'from_state': from_state,
        'to_state': to_state,
        'from_position': state_names[from_state],
        'to_position': state_names[to_state],
        'action': action_names.get((from_state, to_state), f'{from_state}->{to_state}'),
        'executed_reason': execution_row['executed_reason'],
        'signal_QQQ_close': prior_row['qqq_close'],
        'signal_VIX': prior_row['vix_close'],
        'signal_VXN': prior_row['vxn_close'],
    }
    orders = []
    for asset in ('QQQI', 'QQQ', 'TQQQ'):
        old_weight = float(prior_row[f'weight_{asset}'])
        new_weight = float(execution_row[f'weight_{asset}'])
        delta = new_weight - old_weight
        record[f'old_weight_{asset}'] = old_weight
        record[f'new_weight_{asset}'] = new_weight
        record[f'delta_weight_{asset}'] = delta
        if delta > 1e-12:
            orders.append(f'BUY {delta:.0%} {asset}')
        elif delta < -1e-12:
            orders.append(f'SELL {abs(delta):.0%} {asset}')
    record['orders'] = ' | '.join(orders)
    records.append(record)
trade_events = pd.DataFrame(records)
display(trade_events[[
    'signal_date', 'execution_date', 'action', 'orders', 'executed_reason',
    'signal_QQQ_close', 'signal_VIX', 'signal_VXN'
]])


In [ ]:
markers = {
    'Enter QQQ': '^',
    'Add TQQQ leverage': 'P',
    'Reduce TQQQ leverage': 'v',
    'Move to QQQI defense': 'X',
}
ax = overlay['qqq_close'].plot(figsize=(14, 5), title='QQQ close with close-derived signals')
for action, marker in markers.items():
    subset = trade_events[trade_events['action'].eq(action)]
    if not subset.empty:
        ax.scatter(subset['signal_date'], subset['signal_QQQ_close'], marker=marker, s=80, label=action)
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

def plot_execution_points(asset):
    price_column = f'{asset}_open'
    delta_column = f'delta_weight_{asset}'
    buys = trade_events[trade_events[delta_column].gt(1e-12)]
    sells = trade_events[trade_events[delta_column].lt(-1e-12)]
    ax = overlay[price_column].plot(figsize=(14, 4), title=f'{asset} next-open executed trades')
    if not buys.empty:
        ax.scatter(buys['execution_date'], overlay.loc[buys['execution_date'], price_column], marker='^', s=80, label='Buy or increase')
    if not sells.empty:
        ax.scatter(sells['execution_date'], overlay.loc[sells['execution_date'], price_column], marker='v', s=80, label='Sell or reduce')
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()

for asset in ('QQQI', 'QQQ', 'TQQQ'):
    plot_execution_points(asset)


In [ ]:
horizons = (5, 10, 20, 40)
return_columns = ['QQQI_next_open_return', 'QQQ_next_open_return', 'TQQQ_next_open_return']
event_study = trade_events.copy()
for event_index, event in event_study.iterrows():
    location = overlay.index.get_loc(event['execution_date'])
    old_weights = pd.Series({
        'QQQI_next_open_return': event['old_weight_QQQI'],
        'QQQ_next_open_return': event['old_weight_QQQ'],
        'TQQQ_next_open_return': event['old_weight_TQQQ'],
    })
    new_weights = pd.Series({
        'QQQI_next_open_return': event['new_weight_QQQI'],
        'QQQ_next_open_return': event['new_weight_QQQ'],
        'TQQQ_next_open_return': event['new_weight_TQQQ'],
    })
    for horizon in horizons:
        window = overlay.iloc[location:location + horizon][return_columns]
        if len(window) != horizon or window.isna().any().any():
            continue
        old_daily = window.mul(old_weights, axis=1).sum(axis=1)
        new_daily = window.mul(new_weights, axis=1).sum(axis=1)
        old_equity = (1.0 + old_daily).cumprod()
        new_equity = (1.0 + new_daily).cumprod()
        event_study.loc[event_index, f'return_excess_{horizon}d'] = new_equity.iloc[-1] - old_equity.iloc[-1]
        old_dd = old_equity / old_equity.cummax() - 1.0
        new_dd = new_equity / new_equity.cummax() - 1.0
        event_study.loc[event_index, f'drawdown_improvement_{horizon}d'] = new_dd.min() - old_dd.min()
display(event_study[[
    'signal_date', 'execution_date', 'action', 'orders',
    'return_excess_5d', 'return_excess_10d', 'return_excess_20d',
    'return_excess_40d', 'drawdown_improvement_20d'
]])
summary_rows = []
for action, group in event_study.groupby('action'):
    excess = group['return_excess_20d'].dropna()
    dd = group['drawdown_improvement_20d'].dropna()
    summary_rows.append({
        'action': action,
        'events': len(group),
        'mature_20d_events': len(excess),
        'mean_20d_return_excess_pp': excess.mean() * 100,
        'median_20d_return_excess_pp': excess.median() * 100,
        'positive_return_excess_rate_pct': excess.gt(0).mean() * 100,
        'mean_20d_drawdown_improvement_pp': dd.mean() * 100,
        'positive_drawdown_improvement_rate_pct': dd.gt(0).mean() * 100,
    })
signal_effectiveness = pd.DataFrame(summary_rows).sort_values('action')
display(signal_effectiveness)


## Signal-level interpretation

Published twenty-session event study:

- Add TQQQ leverage: 12 events, mean benefit about +5.76 percentage points, median about +4.24 points, and about 91.7% positive relative outcomes.
- QQQI to QQQ: 10 events, mean benefit about +0.74 points and about 50% positive relative outcomes. This is the weakest risk-on transition.
- TQQQ to QQQ: average return sacrifice about 3.40 points, but average twenty-session maximum-drawdown improvement about 6.34 points.
- QQQ to QQQI defense: average return sacrifice about 0.61 points, but average drawdown improvement about 3.97 points.

Entry signals and defensive signals therefore require different evaluation criteria.

In [ ]:
split_frames = []
train_fraction = float(primary_contract['validation']['chronological_train_fraction'])
for key, result in primary_results.items():
    split = chronological_split_metrics(result, train_fraction=train_fraction).reset_index()
    split.insert(0, 'strategy', key)
    split_frames.append(split)
chronological = pd.concat(split_frames, ignore_index=True)
display(chronological)


## Long-history attack-layer validation

QQQI is excluded rather than backfilled. States 0 and 1 map to QQQ; only state 2 receives 75% TQQQ. This validates leverage timing, not the full historical three-asset portfolio.

In [ ]:
long_contract_path = Path('../configs/research_paradigms/qqq_tqqq_vxn_attack_v4_1_long_history.yaml')
long_contract = yaml.safe_load(long_contract_path.read_text(encoding='utf-8'))
long_bars, long_coverage = fetch_adjusted_daily_bars(
    symbols=['QQQ', 'TQQQ', '^VIX', '^VXN'],
    start=long_contract['data']['start_date'],
    end=END_DATE,
)
long_metrics, long_results, long_prepared, long_diagnostics, long_tables = run_attack_layer_comparison(
    long_bars, long_contract
)
display(long_coverage)
display(long_metrics)
display(long_tables['chronological_periods'])
display(long_tables['regime_windows'])
display(long_tables['blocked_vxn_entries'])
long_equity = pd.concat({key: result.daily['equity'] for key, result in long_results.items()}, axis=1)
ax = long_equity.plot(figsize=(13, 5), logy=True, title='2010-2026 attack-layer equity, log scale')
ax.grid(True, alpha=0.3)
plt.show()


## Decision

The leverage transition is the strongest part of v4.1. The weakest measured component is the binary QQQI-to-QQ bridge state, while maximum drawdown remains about -24.43% and turnover increased. The next admissible challenger therefore changes only state-1 allocation to 50% QQQI and 50% QQQ. No signal, threshold, VIX/VXN rule or TQQQ weight changes.